# Bambara SLM: Stage 2 SFT **v3** on Kaggle (resume-capable)

v3 = v2 data without the 45k cap + djelia, back-translation (X→bam), 5,000 Claude-written Bambara conversations, stitched multi-turn chats; loss on **all** assistant turns, LoRA r 64, LR 8e-5, max_len 1024, selection on `eval_sft_loss + eval_chat_loss`.

**Inputs to attach (+ Add Input → Your Work):**
- `bambara-slm-code`: new version from `dist/bambara_slm_colab.zip` (contains `data/sft_v3` and the trainer)
- `bambara-slm-cpt-with-nllb`: the merged CPT model dataset (same as v2)
- *(only to resume a stopped session)* `bambara-slm-sft-v3-run`: dataset made from `resume_pack_sft_v3.zip` (download it from the previous session's output)

Settings: Accelerator **GPU T4**, Internet **On**. Run the setup cell once to check, then **Save Version → Save & Run All (Commit)**.
Expected: ≈ 9,200 steps (≈ 17 h on a T4: 5.8 s/step × 1.17 for longer rows) (1 epoch, 32 examples/step): this needs **two** sessions. Each session stops cleanly at 11 h (exit code 3) and writes `resume_pack_sft_v3.zip`; make it a dataset named `bambara-slm-sft-v3-run` (new version each time), attach it and run again: training resumes from the last checkpoint.


## Setup (one cell)

In [ ]:
import glob, os, shutil, subprocess, time, zipfile
VARIANT   = "v3"
WORK      = "/kaggle/working"
RUN_DIR   = f"{WORK}/runs/sft_{VARIANT}"
MAX_HOURS = 11.0                         # stop cleanly before Kaggle's 12 h limit
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # use ONE T4

def find_input(keyword, required=True):
    hits = [p for p in glob.glob("/kaggle/input/**/", recursive=True) if keyword in p.lower()]
    if not hits:
        assert not required, f"no input matching '{keyword}': click 'Add Input' and attach the dataset"
        return None
    return sorted(hits, key=len)[0].rstrip("/")

def unpack(src, dst):
    """Kaggle input (zip or already-extracted folder) -> writable folder."""
    shutil.rmtree(dst, ignore_errors=True)
    zips = glob.glob(f"{src}/**/*.zip", recursive=True)
    if zips:
        for z in zips:
            zipfile.ZipFile(z).extractall(dst)
    else:
        shutil.copytree(src, dst)
    return dst

# 1. code (+ SFT data)
CODE = unpack(find_input("bambara-slm-code"), "/tmp/code")
CODE = os.path.dirname(sorted(glob.glob(f"{CODE}/**/pyproject.toml", recursive=True), key=len)[0])
assert os.path.exists(f"{CODE}/src/bslm/train/sft.py"), "old bundle: upload the new bambara_slm_colab.zip"
print("code:", CODE, "| SFT data:", os.listdir(f"{CODE}/data/sft_{VARIANT}"))

# 2. base model = merged CPT model (dataset made from results_with_nllb.zip); read in place when already extracted
cpt_in = find_input("bambara-slm-cpt")
cfgs = [p for p in glob.glob(f"{cpt_in}/**/merged/config.json", recursive=True)]
if not cfgs:                                    # still zipped: extract (about 3.4 GB)
    unpack(cpt_in, "/tmp/cpt")
    cfgs = glob.glob("/tmp/cpt/**/merged/config.json", recursive=True)
BASE = os.path.dirname(cfgs[0])
print("base model:", BASE, sorted(os.listdir(BASE)))

# 3. packages
subprocess.run(f'cd "{CODE}" && pip install -q -r requirements-gpu.txt', shell=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

# 4. optional: resume a previous SFT session (dataset made from resume_pack_sft_v3.zip)
run_in = find_input(f"bambara-slm-sft-{VARIANT}-run", required=False)
if run_in:
    tmp = unpack(run_in, "/tmp/prev_sft")
    src = os.path.dirname(sorted(glob.glob(f"{tmp}/**/start_eval.json", recursive=True), key=len)[0])
    shutil.copytree(src, RUN_DIR, dirs_exist_ok=True)
    print("resuming from:", sorted(os.listdir(f"{RUN_DIR}/checkpoints")))
else:
    print("fresh SFT run (no previous run attached)")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

## Train

In [ ]:
os.makedirs(RUN_DIR, exist_ok=True)   # fresh runs: the folder must exist before tee opens train.log
os.environ.update({"PYTHONPATH": f"{CODE}/src", "BSLM_RUN_DIR": RUN_DIR, "BSLM_BASE_MODEL": BASE,
                   "BSLM_MAX_HOURS": str(MAX_HOURS), "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"})
rc = subprocess.call(f'cd "{CODE}" && python -m bslm.train.sft --config configs/train/sft_{VARIANT}.yaml 2>&1 | tee -a "{RUN_DIR}/train.log"; exit ${{PIPESTATUS[0]}}',
                     shell=True, executable="/bin/bash")
print("exit code:", rc, {0: "FINISHED: merged SFT model saved", 2: "finished, but no checkpoint met the +10% limit: tell Claude",
                         3: "time budget reached: download resume_pack_sft and continue in a new session"}.get(rc, "ERROR: see log above"))

## Pack outputs

In [ ]:
import json
os.makedirs(RUN_DIR, exist_ok=True)
!cd "{WORK}/runs" && zip -qr "{WORK}/resume_pack_sft_{VARIANT}.zip" "sft_{VARIANT}" -x "sft_{VARIANT}/merged/*"
if os.path.exists(f"{RUN_DIR}/merged"):
    !cd "{WORK}/runs" && zip -qr "{WORK}/results_sft_{VARIANT}.zip" "sft_{VARIANT}" -x "sft_{VARIANT}/checkpoints/*"
    print(json.dumps(json.load(open(f"{RUN_DIR}/selection.json")), indent=2))
    print(json.dumps(json.load(open(f"{RUN_DIR}/samples.json")), ensure_ascii=False, indent=2))
!ls -lh {WORK}/*.zip